In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_csv('ObesityDataSet_raw_and_data_sinthetic.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

In [ ]:
print('Duplicates:', df.duplicated().sum())

In [ ]:
print(df['NObeyesdad'].value_counts())

plt.figure(figsize=(10,5))
sns.countplot(data=df,x='NObeyesdad')
plt.xticks(rotation=45)
plt.show()

In [ ]:
cat_cols=df.select_dtypes(include='object').columns

encoders={}
for col in cat_cols:
    le=LabelEncoder()
    df[col]=le.fit_transform(df[col])
    encoders[col]=le

df.head()

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(),cmap='coolwarm')
plt.show()

In [ ]:
X=df.drop('NObeyesdad',axis=1)
y=df['NObeyesdad']

In [ ]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [ ]:
pca=PCA(n_components=2)
X_pca=pca.fit_transform(X_scaled)

plt.figure(figsize=(10,6))
plt.scatter(X_pca[:,0],X_pca[:,1],c=y)
plt.colorbar()
plt.show()

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X_scaled,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
svc=SVC(kernel='rbf')
svc.fit(X_train,y_train)

pred=svc.predict(X_test)

print('Accuracy:',accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

In [ ]:
cm=confusion_matrix(y_test,pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.show()

In [ ]:
param_grid={
    'C':[0.1,1,10,100],
    'gamma':['scale','auto',0.01,0.001],
    'kernel':['rbf','poly','sigmoid']
}

grid=GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train,y_train)

print('Best Params:',grid.best_params_)
print('Best Score:',grid.best_score_)

In [ ]:
final_model=grid.best_estimator_
final_model.fit(X_train,y_train)

final_pred=final_model.predict(X_test)

print('Final Accuracy:',accuracy_score(y_test,final_pred))
print(classification_report(y_test,final_pred))

In [ ]:
cm=confusion_matrix(y_test,final_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Greens')
plt.show()

In [ ]:
joblib.dump(final_model,'obesity_svc_model.pkl')
print('Model Saved')